# 01 - Scrape Tanqeeb Job Listings

This notebook is the **scraping** stage of the Tanqeeb pipeline.

- Collects job listing URLs from the Tanqeeb search results (via Selenium, since results are paginated behind JS rendering), starting from page 1 each run and stopping early once it catches up with jobs already scraped.
- Scrapes each individual **new** job page (via `requests` + `BeautifulSoup`) for the relevant fields.
- Saves the full raw history to `tanqeeb_jobs_raw.json` (for auditing), and pushes just this run's new jobs onto a **pending queue** (`tanqeeb_pending_raw_queue.json`) for `02_clean_Tanqeeb.ipynb` to pick up. The queue only grows if cleaning hasn't run yet -- nothing is lost between runs.

Checkpoint files are used throughout so a crash never costs you all the work done so far.


In [11]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException, WebDriverException
from urllib3.exceptions import ReadTimeoutError
from urllib.parse import urljoin
from datetime import datetime
import time
import os
import json
from azure.storage.blob import BlobServiceClient


In [12]:
# ============================================================
# CONFIG
# ============================================================

BASE_URL = "https://saudi.tanqeeb.com"

SEARCH_PATH = "https://saudi.tanqeeb.com/jobs/search"

QUERY_STRING = (
    "keywords=&country=54&state=0&category=-1&"
    "workplace=0&search_period=0&lang=all&change_lang=1"
)

SOURCE_NAME = "Tanqeeb"

# Raw output -- this is the input file for 02_clean_Tanqeeb.ipynb.
# It always holds the FULL cumulative dataset (old jobs + newly found jobs).

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

RAW_JSON_OUTPUT = f"../data/raw/tanqeeb_jobs_raw_{timestamp}.json"
# Persistent master checkpoint: every job ever scraped, across every run.
# This is the source of truth for "which jobs do we already have?".
JOBS_CHECKPOINT = "../data/raw/tanqeeb_jobs_checkpoint.json"

# Discovery checkpoint for the *current* run only: new job URLs that have
# been found during pagination but not scraped yet. Lets a crash during
# pagination or scraping resume without rediscovering/rescraping anything.
NEW_URLS_CHECKPOINT = "../data/raw/tanqeeb_new_urls_checkpoint.json"

# Handoff queue to 02_clean_Tanqeeb.ipynb: only jobs that have been
# scraped but NOT YET cleaned. Each run APPENDS its new jobs here rather
# than overwriting, so nothing is lost even if cleaning lags behind
# scraping by more than one run. 02_clean empties this file once it has
# cleaned everything in it.
PENDING_RAW_QUEUE = "../data/raw/tanqeeb_pending_raw_queue.json"

# How often (in newly scraped jobs) to flush progress to disk
JOBS_SAVE_EVERY = 50

# ---- Incremental scraping ----
# If True: pagination always starts from page 1 (newest listings first)
# and only job URLs NOT already in JOBS_CHECKPOINT are queued for
# scraping; pagination stops early once it looks like we've caught up
# with previously scraped jobs, so re-running this notebook only fetches
# jobs posted since the last run.
# If False: falls back to the original behavior -- every page is walked
# and every listing found is (re-)scraped.
INCREMENTAL_MODE = True

# Stop pagination once this many CONSECUTIVE pages contain zero new job
# URLs (i.e. every job on them was already scraped in a previous run).
# A couple of pages of buffer guards against occasional pinned/duplicate
# listings interrupting an otherwise "all new" or "all old" run.
# Only used when INCREMENTAL_MODE is True.
STOP_AFTER_CONSECUTIVE_KNOWN_PAGES = 2

# Safety cap in case pagination never signals "end" cleanly.
# ~50,700 jobs / ~28 per page is roughly 1,800+ pages, so raise
# this if you actually intend to (re)scrape the entire site.
#MAX_PAGES = 2000
MAX_PAGES = 1
# Stop pagination after this many consecutive driver failures in a row
MAX_CONSECUTIVE_FAILURES = 5


In [13]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================



def build_page_url(page_num):
    """
    Tanqeeb paginates using a path segment, not a query param:
      Page 1: https://saudi.tanqeeb.com/jobs/search?keywords=...
      Page 2: https://saudi.tanqeeb.com/jobs/search/page/2?keywords=...
      Page N: https://saudi.tanqeeb.com/jobs/search/page/N?keywords=...
    """
    if page_num <= 1:
        return f"{SEARCH_PATH}?{QUERY_STRING}"
    return f"{SEARCH_PATH}/page/{page_num}?{QUERY_STRING}"


def get_job_links_from_html(html):
    soup = BeautifulSoup(html, "html.parser")
    return soup.select("h2.search-job-title a.search-job-title-link")


def make_driver():
    """Create a fresh headless Chrome driver with a longer command timeout."""
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-blink-features=AutomationControlled")
    d = webdriver.Chrome(options=options)
    # Default Selenium command timeout is 120s; bump it since some
    # search pages render slowly under headless Chrome after many navigations.
    d.command_executor.set_timeout(300)
    return d


def load_jobs_checkpoint():
    """Persistent master list of every job scraped so far, across all runs."""
    if os.path.exists(JOBS_CHECKPOINT):
        with open(JOBS_CHECKPOINT, "r", encoding="utf-8") as f:
            return json.load(f)
    return []


def save_jobs_checkpoint(jobs):
    os.makedirs(os.path.dirname(JOBS_CHECKPOINT), exist_ok=True)
    with open(JOBS_CHECKPOINT, "w", encoding="utf-8") as f:
        json.dump(jobs, f, ensure_ascii=False, indent=2)


def load_known_job_urls(jobs):
    """Set of job URLs already present in the master job list."""
    return {
        job.get("url")
        for job in jobs
        if job.get("url") and job.get("url") != "N/A"
    }


def load_new_urls_checkpoint():
    """Resume an in-progress discovery/scrape run: new URLs not yet scraped."""
    if os.path.exists(NEW_URLS_CHECKPOINT):
        with open(NEW_URLS_CHECKPOINT, "r", encoding="utf-8") as f:
            return json.load(f).get("new_urls", [])
    return []


def save_new_urls_checkpoint(new_urls):
    os.makedirs(os.path.dirname(NEW_URLS_CHECKPOINT), exist_ok=True)
    with open(NEW_URLS_CHECKPOINT, "w", encoding="utf-8") as f:
        json.dump({"new_urls": new_urls}, f, ensure_ascii=False, indent=2)


def clear_new_urls_checkpoint():
    if os.path.exists(NEW_URLS_CHECKPOINT):
        os.remove(NEW_URLS_CHECKPOINT)


def load_json_list(path):
    """Load a JSON file containing a list of job dicts; [] if it doesn't exist yet."""
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return []


def save_json_list(path, items):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(items, f, ensure_ascii=False, indent=2)


## Step 1: Load what we already have, then discover NEW job URLs

In [14]:
# Master list of every job ever scraped (persists across runs)
all_jobs = load_jobs_checkpoint()
known_job_urls = load_known_job_urls(all_jobs)

print(f"Jobs already in the dataset: {len(all_jobs)}")

if INCREMENTAL_MODE:
    print("Incremental mode ON: pagination starts from page 1 and only "
          "newly posted jobs will be queued for scraping.")
else:
    print("Incremental mode OFF: doing a full scrape of every listing found.")

# Resume an in-progress discovery run if one was interrupted (crash-safety),
# otherwise start pagination fresh from page 1 -- newest listings are shown
# first, so page 1 is where new postings will appear.
new_job_urls = load_new_urls_checkpoint()

if new_job_urls:
    print(f"Resuming an interrupted run: {len(new_job_urls)} new URL(s) "
          f"already discovered and pending scrape.")

driver = make_driver()
consecutive_failures = 0
consecutive_known_pages = 0
page_num = 1

try:
    while page_num <= MAX_PAGES:

        page_url = build_page_url(page_num)

        print(f"\nLoading search page {page_num}: {page_url}")

        try:
            driver.get(page_url)
            # Wait for page to load / JS to render results
            time.sleep(5)
            html = driver.page_source
            consecutive_failures = 0
        except (TimeoutException, WebDriverException, ReadTimeoutError) as e:
            consecutive_failures += 1
            print(f"  !! Driver error on page {page_num} "
                  f"(failure {consecutive_failures}/{MAX_CONSECUTIVE_FAILURES}): {e}")

            # Kill and restart the browser process; retry the same page
            try:
                driver.quit()
            except Exception:
                pass
            time.sleep(3)
            driver = make_driver()

            if consecutive_failures >= MAX_CONSECUTIVE_FAILURES:
                print("Too many consecutive driver failures. Stopping pagination "
                      "(progress so far is saved).")
                break

            continue  # retry page_num with the fresh driver, don't advance

        job_links = get_job_links_from_html(html)

        print(f"  -> Job links found on this page: {len(job_links)}")

        # Stop condition: no job cards on this page means we've
        # gone past the last page of results.
        if not job_links:
            print("No more job listings found. Stopping pagination.")
            break

        page_had_new_url = False
        new_on_this_page = 0

        for link in job_links:
            href = link.get("href")
            if not href:
                continue

            full_url = urljoin(BASE_URL, href)
            already_known = full_url in known_job_urls
            already_queued = full_url in new_job_urls

            if already_known and INCREMENTAL_MODE:
                # Already scraped in a previous run -- nothing to do.
                continue

            if not already_queued:
                new_job_urls.append(full_url)
                page_had_new_url = True
                new_on_this_page += 1

        if INCREMENTAL_MODE:
            if new_on_this_page == 0:
                consecutive_known_pages += 1
                print(f"  -> No new jobs on this page "
                      f"({consecutive_known_pages}/{STOP_AFTER_CONSECUTIVE_KNOWN_PAGES} "
                      f"consecutive already-known pages).")
                if consecutive_known_pages >= STOP_AFTER_CONSECUTIVE_KNOWN_PAGES:
                    print("Caught up with previously scraped jobs. Stopping pagination.")
                    break
            else:
                consecutive_known_pages = 0
                print(f"  -> New jobs found on this page: {new_on_this_page}")
        else:
            # Original (non-incremental) safety check: if a page returns
            # only links we've already queued (e.g. the site clamped an
            # out-of-range page back to the last page), treat that as the end.
            if not page_had_new_url:
                print("No new job URLs on this page (likely repeated last page). Stopping.")
                break

        # Persist discovery progress after every page
        save_new_urls_checkpoint(new_job_urls)

        page_num += 1

finally:
    driver.quit()

print(f"\nNewly discovered job URLs to scrape this run: {len(new_job_urls)}")


Jobs already in the dataset: 550
Incremental mode ON: pagination starts from page 1 and only newly posted jobs will be queued for scraping.
Resuming an interrupted run: 10004 new URL(s) already discovered and pending scrape.


KeyboardInterrupt: 

## Step 2: Scrape each newly discovered job page

In [ ]:
# Guard against re-scraping anything that (somehow) is already in the
# master list, and track how many genuinely new jobs this run adds.
already_scraped_urls = {job.get("url") for job in all_jobs}
jobs_added_this_run = 0
new_jobs_this_run = []  # only the jobs added in THIS run, for handoff to 02_clean

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/140.0.0.0 Safari/537.36"
    )
}

for i, url in enumerate(new_job_urls, start=1):

    if url in already_scraped_urls:
        continue

    print(f"\n[{i}/{len(new_job_urls)}] Scraping:")
    print(url)

    try:
        response = requests.get(url, headers=headers, timeout=30)

        if response.status_code != 200:
            print("Failed:", response.status_code)
            continue

        # Keep the full raw HTML of the page itself -- this is the ultimate
        # source of truth. Even if a selector below misses something (or the
        # site changes its markup), the untouched page is preserved here so
        # nothing is ever permanently lost at the scraping stage.
        raw_html = response.text

        soup = BeautifulSoup(response.text, "html.parser")

        # JOB ID
        apply_button = soup.select_one(".apply-btn")
        job_id = apply_button.get("data-job-id") if apply_button else None
        apply_url = None
        if apply_button:
            apply_url = apply_button.get("href") or apply_button.get("data-url")

        # JOB TITLE
        title_element = soup.select_one("h3.job-title-text")
        job_title = title_element.get_text(" ", strip=True) if title_element else None

        # COMPANY NAME
        company_element = soup.select_one(
            "a.job-meta-company, .job-company-name, .company-name, a.company-name-link"
        )
        company_name = (
            company_element.get_text(" ", strip=True)
            if company_element else None
        )
        # Raw href too, if the company element is a link -- cheap to grab
        # and useful later for joining/deduping companies.
        company_url = (
            urljoin(BASE_URL, company_element.get("href"))
            if company_element and company_element.get("href")
            else None
        )
        company_logo_element = soup.select_one(".company-logo img, .job-company-logo img")
        company_logo_url = (
            urljoin(BASE_URL, company_logo_element.get("src"))
            if company_logo_element and company_logo_element.get("src")
            else None
        )

        # WORK MODE + JOB TYPE
        # Every tag on the job card, raw and unclassified -- kept as-is so
        # nothing is dropped even if the site adds tag values we don't
        # recognize below.
        tag_elements = soup.select(".job-tags .job-tag")
        all_tags = [
            t.get_text(" ", strip=True)
            for t in tag_elements
            if t.get_text(strip=True)
        ]

        work_mode = None
        job_type = None

        for value in all_tags:
            value_lower = value.lower()

            if value_lower in ["on-site", "remote", "hybrid"]:
                work_mode = value
            elif value_lower in [
                "full time", "part time", "contract",
                "internship", "temporary", "freelance"
            ]:
                job_type = value

        # LOCATION
        location_element = soup.select_one(".job-meta-item span")
        location = (
            location_element.get_text(" ", strip=True)
            if location_element
            else None
        )

        # POSTED DATE
        date_element = soup.select_one(".job-date")
        raw_posted_date = date_element.get("data-datetime") if date_element else None

        parsed_posted_date = (
            pd.to_datetime(raw_posted_date, errors="coerce")
            if raw_posted_date
            else None
        )

        if parsed_posted_date is not None and not pd.isna(parsed_posted_date):
            # Convert pandas Timestamp to a normal JSON-serializable string
            posted_date = parsed_posted_date.isoformat()
        else:
            posted_date = None

        # JOB DETAILS -- capture EVERY label/value row the page has, not just
        # the ones we happen to promote to top-level columns below. Whatever
        # extra fields Tanqeeb adds or removes over time show up here
        # automatically.
        job_details = {}
        meta_rows = soup.select(".meta-data .meta")

        for row in meta_rows:
            spans = row.find_all("span")
            if len(spans) >= 2:
                label = spans[0].get_text(" ", strip=True)
                value = spans[1].get_text(" ", strip=True)
                job_details[label] = value

        experience = job_details.get("Experience")
        employment = job_details.get("Employment")
        gender = job_details.get("Gender")
        education = job_details.get("Education")
        industry = job_details.get("Industry")
        career_level = job_details.get("Career Level")
        salary = job_details.get("Salary")

        # SKILLS
        skill_elements = soup.select(".job-skills .skill, .skills-list .skill-tag")
        skills_list = [
            s.get_text(" ", strip=True)
            for s in skill_elements
            if s.get_text(strip=True)
        ]
        skills = "; ".join(skills_list) if skills_list else None

        # JOB DESCRIPTION
        description_element = soup.select_one("#jobDescriptionBody")
        description = (
            description_element.get_text(" ", strip=True)
            if description_element else None
        )

        # Full visible page text, as a last-resort raw fallback -- lets
        # 02_clean (or any future re-parse) recover text even for fields we
        # have no selector for yet.
        full_page_text = soup.get_text(" ", strip=True)

        # SAVE JOB -- curated convenience fields, PLUS the complete raw
        # capture (raw_html, full_page_text, job_details_raw, all_tags) so
        # nothing scraped from the page is ever discarded at this stage.
        job = {
            "id": job_id,
            "source": SOURCE_NAME,
            "title": job_title,
            "company": company_name,
            "company_url": company_url,
            "company_logo_url": company_logo_url ,
            "description": description ,
            "url": url ,
            "apply_url": apply_url,
            "location": location ,
            "work_mode": work_mode,
            "job_type": employment if employment else job_type,
            "career_level": career_level,
            "experience": experience,
            "education": education,
            "gender": gender,
            "industry": industry,
            "skills": skills,
            "salary": salary ,
            "posted_date": posted_date,
            "collected_at": datetime.now().isoformat(),
            # # ---- complete raw capture (nothing filtered out) ----
            # "tags_raw": all_tags,
            # "job_details_raw": job_details,
            # "full_page_text": full_page_text,
            # "raw_html": raw_html,
        }

        all_jobs.append(job)
        new_jobs_this_run.append(job)
        already_scraped_urls.add(url)
        jobs_added_this_run += 1

        print("Scraped:", job_title)

        # Periodically flush progress so a crash doesn't lose everything
        if jobs_added_this_run % JOBS_SAVE_EVERY == 0:
            save_jobs_checkpoint(all_jobs)
            print(f"  (checkpoint saved: {len(all_jobs)} jobs total, "
                  f"{jobs_added_this_run} new this run)")

    except Exception as e:
        print("Error:", e)

    # Wait between requests
    time.sleep(1)

# Final checkpoint save after the loop completes. Every discovered URL has
# now been handled (scraped or errored), so clear the discovery checkpoint
# for next time.
save_jobs_checkpoint(all_jobs)
clear_new_urls_checkpoint()

print(f"\nNew jobs added this run: {jobs_added_this_run}")
print(f"Total jobs in dataset: {len(all_jobs)}")



[551/10004] Scraping:
https://saudi.tanqeeb.com/jobs-in-middle-east/all/jobs/021250596.html
Scraped: Retail Center

[552/10004] Scraping:
https://saudi.tanqeeb.com/jobs-in-middle-east/all/jobs/021250595.html
Scraped: Room Service Manager

[553/10004] Scraping:
https://saudi.tanqeeb.com/jobs-in-middle-east/all/jobs/021250594.html
Scraped: Sales Manager (Saudi Only)

[554/10004] Scraping:
https://saudi.tanqeeb.com/jobs-in-middle-east/all/jobs/021250593.html
Scraped: Sales Manager

[555/10004] Scraping:
https://saudi.tanqeeb.com/jobs-in-middle-east/all/jobs/021250592.html


KeyboardInterrupt: 

## Step 3: Save raw data -- full history, plus a queue of only the NEW jobs

In [ ]:
# 1) Full historical raw mirror (every job ever scraped) -- useful for
#    auditing/debugging, not used directly by the rest of the pipeline.
save_json_list(RAW_JSON_OUTPUT, all_jobs)

# 2) Hand off ONLY this run's new jobs to 02_clean_Tanqeeb.ipynb via a
#    pending queue. We APPEND (not overwrite) so nothing is lost if
#    02_clean hasn't been run since the last scrape.
pending_raw = load_json_list(PENDING_RAW_QUEUE)
pending_urls = {j.get("url") for j in pending_raw}

for job in new_jobs_this_run:
    if job.get("url") not in pending_urls:
        pending_raw.append(job)
        pending_urls.add(job.get("url"))

save_json_list(PENDING_RAW_QUEUE, pending_raw)

print("\n" + "=" * 60)
print("New jobs found this run:", jobs_added_this_run)
print("Total jobs in full raw history:", len(all_jobs))
print("Jobs now waiting to be cleaned (pending queue):", len(pending_raw))
print("Raw history saved as:", RAW_JSON_OUTPUT)
print("Pending queue saved as:", PENDING_RAW_QUEUE)
print("=" * 60)



New jobs found this run: 27
Total jobs in full raw history: 27
Jobs now waiting to be cleaned (pending queue): 27
Raw history saved as: ../data/raw/tanqeeb_jobs_raw.json
Pending queue saved as: ../data/raw/tanqeeb_pending_raw_queue.json
